## Content Based Filtering

### Loading Dataset

In [8]:
import pandas as pd 

movies_df = pd.read_csv("../data/ml-latest-small/movies.csv")
tags_df = pd.read_csv("../data/ml-latest-small/tags.csv")

### Understanding the data

In [9]:
def understanding_data(df):
    print("DataFrame Info:")
    print(df.info())
    print("\nDataFrame Head:")
    print(df.head())
    print(f"\nDataFrame Shape: {df.shape}")
    print(f"\nDataFrame Data Types:\n{df.dtypes}")
    print("\nMissing Values:")
    print(df.isnull().sum())
    print("\nUnique Values per Column:")
    for column in df.columns:
        unique_values = df[column].nunique()
        print(f"{column}: {unique_values} unique values")

In [10]:
understanding_data(movies_df)

DataFrame Info:
<class 'pandas.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  9742 non-null   int64
 1   title    9742 non-null   str  
 2   genres   9742 non-null   str  
dtypes: int64(1), str(2)
memory usage: 626.1 KB
None

DataFrame Head:
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  

DataFrame Shape: (9742, 3)

DataFrame Data 

In [11]:
understanding_data(tags_df)

DataFrame Info:
<class 'pandas.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   userId     3683 non-null   int64
 1   movieId    3683 non-null   int64
 2   tag        3683 non-null   str  
 3   timestamp  3683 non-null   int64
dtypes: int64(3), str(1)
memory usage: 151.7 KB
None

DataFrame Head:
   userId  movieId              tag   timestamp
0       2    60756            funny  1445714994
1       2    60756  Highly quotable  1445714996
2       2    60756     will ferrell  1445714992
3       2    89774     Boxing story  1445715207
4       2    89774              MMA  1445715200

DataFrame Shape: (3683, 4)

DataFrame Data Types:
userId       int64
movieId      int64
tag            str
timestamp    int64
dtype: object

Missing Values:
userId       0
movieId      0
tag          0
timestamp    0
dtype: int64

Unique Values per Column:
userId: 58 unique values
movieId: 1572 unique

In [16]:
movies_df['genres'].value_counts()

np.int64(9742)

In [14]:
tags_df['tag'].value_counts()

tag
In Netflix queue     131
atmospheric           36
superhero             24
thought-provoking     24
funny                 23
                    ... 
for katie              1
austere                1
gun fu                 1
heroic bloodshed       1
Heroic Bloodshed       1
Name: count, Length: 1589, dtype: int64

### Data  Cleaning

In [26]:
movies_df['genres_clean'] = movies_df['genres'].str.replace('|', ' ', regex=False)
movies_df['genres_clean'] = movies_df['genres_clean'].str.replace('(no genres listed)', '', regex=False)

In [28]:
movies_df[movies_df['genres_clean']=='']

,movieId,title,genres,genres_clean
8517,114335,La cravate (1957),(no genres listed),
8684,122888,Ben-hur (2016),(no genres listed),
8687,122896,Pirates of the Caribbean: Dead Men Tell No Tal...,(no genres listed),
8782,129250,Superfast! (2015),(no genres listed),
8836,132084,Let It Be Me (1995),(no genres listed),
8902,134861,Trevor Noah: African American (2013),(no genres listed),
9033,141131,Guardians (2016),(no genres listed),
9053,141866,Green Room (2015),(no genres listed),
9070,142456,The Brand New Testament (2015),(no genres listed),
9091,143410,Hyena Road,(no genres listed),


In [29]:
tags_df

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200
...,...,...,...,...
3678,606,7382,for katie,1171234019
3679,606,7936,austere,1173392334
3680,610,3265,gun fu,1493843984
3681,610,3265,heroic bloodshed,1493843978


### Joining all tags together so that each movie has only 1 row

In [30]:
movie_tags = tags_df.groupby('movieId', as_index=False)['tag'].agg(' '.join)

In [31]:
movie_tags

,movieId,tag
0,1,pixar pixar fun
1,2,fantasy magic board game Robin Williams game
2,3,moldy old
3,5,pregnancy remake
4,7,remake
...,...,...
1567,183611,Comedy funny Rachel McAdams
1568,184471,adventure Alicia Vikander video game adaptation
1569,187593,Josh Brolin Ryan Reynolds sarcasm
1570,187595,Emilia Clarke star wars


### Joining movies & Tags data together

In [32]:
movies_df = movies_df.merge(movie_tags, on='movieId', how='left')
movies_df['tag'] = movies_df['tag'].fillna('')
movies_df['content'] = movies_df['genres_clean'] + ' ' + movies_df['tag']

In [33]:
movies_df

,movieId,title,genres,genres_clean,tag,content
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy,pixar pixar fun,Adventure Animation Children Comedy Fantasy pi...
1,2,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy,fantasy magic board game Robin Williams game,Adventure Children Fantasy fantasy magic board...
2,3,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance,moldy old,Comedy Romance moldy old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance,,Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,Comedy,pregnancy remake,Comedy pregnancy remake
...,...,...,...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy,Action Animation Comedy Fantasy,,Action Animation Comedy Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy,Animation Comedy Fantasy,,Animation Comedy Fantasy
9739,193585,Flint (2017),Drama,Drama,,Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation,Action Animation,,Action Animation


### lowering all the content

In [34]:
movies_df['content'] = movies_df['content'].apply(lambda x: x.lower())

### NLP Techniques Start

In [36]:
import nltk 

In [37]:
from nltk.stem.porter import PorterStemmer
ps= PorterStemmer()

### Stemming Content

In [38]:
def stem(text):
    y=[]
    
    
    for i in text.split():
        y.append(ps.stem(i))
        
    return " ".join(y)

In [39]:
ps.stem('loved')

'love'

In [40]:
ps.stem(movies_df['content'][0])

'adventure animation children comedy fantasy pixar pixar fun'

In [41]:
movies_df['content'] = movies_df['content'].apply(stem)

In [42]:
movies_df['content']

0       adventur anim children comedi fantasi pixar pi...
1       adventur children fantasi fantasi magic board ...
2                                 comedi romanc moldi old
3                                     comedi drama romanc
4                                   comedi pregnanc remak
                              ...                        
9737                           action anim comedi fantasi
9738                                  anim comedi fantasi
9739                                                drama
9740                                          action anim
9741                                               comedi
Name: content, Length: 9742, dtype: str

### TF-IDF Vectorization

In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer 
cv = TfidfVectorizer (max_features=5000,stop_words='english')

In [52]:
vectors = cv.fit_transform(movies_df['content']).toarray()

In [53]:
vectors

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(9742, 1582))

### Calculating Cosine Similarity

In [54]:
from sklearn.metrics.pairwise import cosine_similarity

In [55]:
similarity = cosine_similarity(vectors)

In [56]:
sorted(list(enumerate(similarity[0])),reverse=True,key=lambda x:x[1])[1:6]

[(1757, np.float64(0.8622460468679132)),
 (2355, np.float64(0.6439312998581017)),
 (8695, np.float64(0.36769350644505683)),
 (1706, np.float64(0.3576249377353534)),
 (2809, np.float64(0.3576249377353534))]

### Creating the recommender

In [57]:
def recommend(movie):
    movie_index = movies_df[movies_df['title']==movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]
    for i in movies_list:
        print(movies_df.iloc[i[0]].title)

In [58]:
recommend('Toy Story (1995)')

Bug's Life, A (1998)
Toy Story 2 (1999)
Guardians of the Galaxy 2 (2017)
Antz (1998)
Adventures of Rocky and Bullwinkle, The (2000)
